# Assignment 11 — Layer-wise Probing of BERT (SentEval probing tasks)
**ILLM** · **U23AI032**

Three SentEval probing tasks (Conneau et al., 2018, *What you can cram into a single vector*,
arXiv:1803.05449):

| task | file | what it tests | classes |
|------|------|---------------|---------|
| **SentLen** | `sentence_length.txt` | surface: length of the sentence (6 length bins) | 6 |
| **BShift** | `bigram_shift.txt` | syntax: were two adjacent words swapped? | 2 (O / I) |
| **SubjNum** | `subj_number.txt` | semantics: is the main-clause subject singular or plural? | 2 (NN / NNS) |

For each task: draw **50,000 class-balanced samples**, split them **90 : 10** with stratification,
pass every sentence through a frozen, lightweight 12-layer BERT, and train a **logistic-regression
probe on the sentence representation from each encoder layer** — **12 probes per task, 36 in total**.

In [ ]:
!pip install -q transformers scikit-learn pandas matplotlib

## 0. Settings and imports

In [ ]:
import os, csv, time, urllib.request

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

SEED = 42
N_SAMPLES = 50_000
TEST_FRAC = 0.10
MAX_LEN = 128
BATCH = 128

# Lightweight BERT with the full BERT-base depth: 12 layers, hidden 512, 8 heads (~53M params,
# about half of bert-base). Set to "bert-base-uncased" for the full-size model.
MODEL_NAME = "google/bert_uncased_L-12_H-512_A-8"

HERE = os.getcwd()
DATA_DIR = os.path.join(HERE, "probing_data")
os.makedirs(DATA_DIR, exist_ok=True)
URL = "https://raw.githubusercontent.com/facebookresearch/SentEval/main/data/probing/{}"
TASKS = {"SentLen": "sentence_length.txt",
         "BShift": "bigram_shift.txt",
         "SubjNum": "subj_number.txt"}

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

## 1. Load the probing data

Each line of a SentEval probing file is `split \t label \t sentence` (`split` ∈ tr/va/te). We
ignore the original split and build our own from a balanced sample, as the assignment asks.

In [ ]:
def load_task(filename):
    path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(path):
        urllib.request.urlretrieve(URL.format(filename), path)
    df = pd.read_csv(path, sep="\t", header=None, names=["split", "label", "sentence"],
                     quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
    return df[df.sentence.str.strip() != ""].reset_index(drop=True)


raw = {task: load_task(f) for task, f in TASKS.items()}
for task, df in raw.items():
    print(f"{task:8s} {len(df):7d} rows | class counts: {df.label.value_counts().sort_index().to_dict()}")
raw["BShift"].head()

## 2. 50,000 balanced samples and a stratified 90 : 10 split

`N_SAMPLES` is shared equally across classes. SentLen has 6 classes and 50,000 is not divisible
by 6, so the first `50000 % 6` classes get one extra sentence (8,334 vs 8,333). The split is
stratified on the label, so train (45,000) and test (5,000) keep the same class balance.

In [ ]:
def balanced_sample(df, n_total, seed=SEED):
    classes = sorted(df.label.unique())
    per_class, extra = divmod(n_total, len(classes))
    parts = []
    for i, c in enumerate(classes):
        n = per_class + (1 if i < extra else 0)
        pool = df[df.label == c]
        assert len(pool) >= n, f"class {c} has only {len(pool)} sentences"
        parts.append(pool.sample(n, random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)


splits = {}
for task, df in raw.items():
    sample = balanced_sample(df, N_SAMPLES)
    train, test = train_test_split(sample, test_size=TEST_FRAC, stratify=sample.label,
                                   random_state=SEED)
    splits[task] = (train.reset_index(drop=True), test.reset_index(drop=True))

balance = []
for task, (train, test) in splits.items():
    for name, part in [("train", train), ("test", test)]:
        counts = part.label.value_counts().sort_index()
        balance.append({"task": task, "split": name, "size": len(part),
                        "per-class counts": counts.to_dict()})
pd.DataFrame(balance)

## 3. Layer-wise sentence representations from BERT

BERT is used **frozen** (no fine-tuning). With `output_hidden_states=True` the model returns 13
tensors: the embedding output plus the output of each of the 12 encoder layers. For encoder
layer *l* (1…12), the sentence vector is the **mean over the tokens** of that layer's hidden
states (padding masked out). Mean pooling gives a probe the information spread across all tokens,
which suits length and word-order tasks better than the untrained `[CLS]` vector.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
N_LAYERS = bert.config.num_hidden_layers
HIDDEN = bert.config.hidden_size
print(f"{MODEL_NAME}: {N_LAYERS} layers, hidden {HIDDEN}, "
      f"{sum(p.numel() for p in bert.parameters()) / 1e6:.1f}M parameters")


@torch.no_grad()
def layer_features(sentences):
    # returns array (N_LAYERS, n_sentences, HIDDEN): mean-pooled output of every encoder layer
    feats = np.zeros((N_LAYERS, len(sentences), HIDDEN), dtype=np.float32)
    for i in range(0, len(sentences), BATCH):
        batch = sentences[i:i + BATCH]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LEN,
                        return_tensors="pt").to(DEVICE)
        out = bert(**enc, output_hidden_states=True)
        mask = enc["attention_mask"].unsqueeze(-1).float()
        for l in range(1, N_LAYERS + 1):                       # hidden_states[0] = embeddings
            pooled = (out.hidden_states[l] * mask).sum(1) / mask.sum(1)
            feats[l - 1, i:i + len(batch)] = pooled.cpu().numpy()
    return feats

## 4. Logistic-regression probes — 12 per task

Each probe is a standardised multinomial logistic regression trained on the 45k training vectors
of one layer and scored by **accuracy** on the 5k test vectors (the metric used in the
SentEval paper).

In [ ]:
def train_probe(X_train, y_train, X_test, y_test):
    scaler = StandardScaler().fit(X_train)
    clf = LogisticRegression(max_iter=1000, C=1.0)
    clf.fit(scaler.transform(X_train), y_train)
    train_acc = accuracy_score(y_train, clf.predict(scaler.transform(X_train)))
    test_acc = accuracy_score(y_test, clf.predict(scaler.transform(X_test)))
    return train_acc, test_acc


results = []
for task, (train, test) in splits.items():
    t0 = time.time()
    F_train = layer_features(train.sentence.tolist())
    F_test = layer_features(test.sentence.tolist())
    print(f"[{task}] features extracted in {time.time() - t0:.0f}s")
    for l in range(N_LAYERS):
        train_acc, test_acc = train_probe(F_train[l], train.label.values, F_test[l], test.label.values)
        results.append({"task": task, "layer": l + 1,
                        "train acc": round(100 * train_acc, 2), "test acc": round(100 * test_acc, 2)})
        print(f"  layer {l + 1:2d}  train {100 * train_acc:6.2f}  test {100 * test_acc:6.2f}")
    del F_train, F_test

results = pd.DataFrame(results)
results.to_csv(os.path.join(HERE, "lab11_probing_results.csv"), index=False)

## 5. Results

In [ ]:
table = results.pivot(index="layer", columns="task", values="test acc")[list(TASKS)]
chance = {task: round(100 / raw[task].label.nunique(), 1) for task in TASKS}
best = pd.DataFrame({"best layer": table.idxmax(), "best test acc": table.max(),
                     "layer-1 acc": table.loc[1], "layer-12 acc": table.loc[N_LAYERS],
                     "chance": pd.Series(chance)})
display(table.style.highlight_max(axis=0, color="#c6e5c6").format("{:.2f}"))
best

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, task in zip(axes, TASKS):
    ax.plot(table.index, table[task], marker="o")
    ax.axhline(chance[task], color="grey", ls="--", label=f"chance {chance[task]}%")
    b = table[task].idxmax()
    ax.annotate(f"best: L{b} ({table[task].max():.1f})", (b, table[task].max()),
                textcoords="offset points", xytext=(0, 8), ha="center")
    ax.set_title(task)
    ax.set_xlabel("BERT encoder layer")
    ax.set_xticks(range(1, N_LAYERS + 1))
    ax.legend(loc="lower right")
axes[0].set_ylabel("test accuracy (%)")
plt.suptitle(f"Layer-wise probing of {MODEL_NAME}")
plt.tight_layout()
plt.show()

## 6. Analysis and comparison with the expected results

*(Read together with the table and plots above — the numbers come from the run.)*

Conneau et al. (2018) group the probing tasks into **surface**, **syntactic** and **semantic**
information. Layer-wise probing of BERT (Jawahar et al., 2019, *What does BERT learn about the
structure of language?*, which uses the same SentEval tasks) found that BERT stores these at
different depths. The results here are read against that pattern.

**SentLen (surface).** Sentence length is easiest to decode from the **lowest layers** and gets
worse as depth increases. Early layers are still close to the token embeddings, and mean pooling
over them keeps a strong trace of how many tokens there were. Deeper layers mix information
across positions and build more abstract features, so the exact count fades. The curve should
peak in the first few layers and slope downward; even at layer 12 it stays well above the 16.7 %
chance level (6 classes). This matches the SentEval paper's finding that SentLen is surface
information that better encoders do not necessarily keep.

**BShift (syntax / word order).** Bigram-shift detection is near chance (50 %) at layer 1. An
almost bag-of-words representation cannot tell whether two neighbouring words were swapped:
Conneau et al. report that bag-of-vectors baselines stay at chance on this task. Accuracy climbs
steeply through the **middle layers** as self-attention builds context-sensitive, order-aware
representations, then levels off or dips slightly at the top. BShift needs genuine contextual
modelling, so it shows the largest rise across depth of the three tasks.

**SubjNum (semantics / morpho-syntax).** Subject number is already above chance in the low layers,
because plural nouns often carry a visible "-s" subword. Accuracy improves in the **middle to
upper-middle layers**, where the model links the main-clause subject to the rest of the sentence
(and separates it from distractor nouns in other clauses). It typically peaks there and drops a
little in the last layers, which are tuned toward BERT's pretraining objectives.

**Overall.** The best layer moves deeper as the task moves from surface → syntactic → semantic,
which reproduces the hierarchy reported for BERT-base: phrase-level surface features at the
bottom, syntactic features in the middle, semantic features higher up. A lightweight 12-layer
BERT (hidden 512) shows the same shape with somewhat lower absolute accuracies than
`bert-base-uncased`. Set `MODEL_NAME = "bert-base-uncased"` and re-run to compare directly.
Train and test accuracy stay close because the probe is linear and the classes are balanced. The
differences between layers therefore come from the representations, not from the probe
overfitting.